In [2]:
import yfinance as yf
import pandas as pd
import sqlite3

# Tickers we care about (Mag 7 + benchmark)
tickers = ["AAPL", "MSFT", "GOOGL", "AMZN", "META", "NVDA", "TSLA", "VOO"]

# Pull all of 2025
raw = yf.download(tickers, start="2025-01-01", end="2026-01-01")

raw.head()

[*********************100%***********************]  8 of 8 completed


Price            Close                                                  \
Ticker            AAPL        AMZN       GOOGL        META        MSFT   
Date                                                                     
2025-01-02  242.525162  220.220001  188.558258  596.845276  414.568604   
2025-01-03  242.037827  224.190002  190.907394  602.213745  419.292908   
2025-01-06  243.668915  227.610001  195.964020  627.681641  423.749756   
2025-01-07  240.894073  222.110001  194.590378  615.420776  418.322266   
2025-01-08  241.381409  222.130005  193.057465  608.279480  420.491241   

Price                                                 High              ...  \
Ticker            NVDA        TSLA         VOO        AAPL        AMZN  ...   
Date                                                                    ...   
2025-01-02  138.264709  379.279999  529.258667  247.746638  225.149994  ...   
2025-01-03  144.422684  410.440002  536.092834  242.853364  225.360001  ...   
2025-01-06  149.381042  411.049988  539.214417  245.986258  228.839996  ...   
2025-01-07  140.094086  394.359985  533.148376  244.215924  228.380005  ...   
2025-01-08  140.064102  394.940002  533.867188  242.385931  223.520004  ...   

Price             Open                Volume                                \
Ticker            TSLA         VOO      AAPL      AMZN     GOOGL      META   
Date                                                                         
2025-01-02  390.100006  533.749081  55740700  33956600  20370800  12682300   
2025-01-03  381.480011  531.947055  40244100  27515600  18596200  11436800   
2025-01-06  423.200012  539.913604  45045600  31849800  29563600  14560800   
2025-01-07  405.829987  540.937708  40856000  28084200  26487200  12071500   
2025-01-08  392.950012  533.059693  37628900  25033300  24864800  10085800   

Price                                                
Ticker          MSFT       NVDA       TSLA      VOO  
Date                                                 
2025-01-02  16896500  198247200  109710700  7142700  
2025-01-03  16662900  229322500   95423300  6416300  
2025-01-06  20573600  265377400   85516500  5983400  
2025-01-07  18139100  351782200   75699500  5383200  
2025-01-08  15054600  227349900   73038800  4317000  

[5 rows x 40 columns]

In [3]:
# Reshape from wide multi-index to long format: one row per (date, ticker)
prices = raw.stack(level=1, future_stack=True).reset_index()
prices.columns = ["date", "ticker", "close", "high", "low", "open", "volume"]

# Keep only what we need, in a sensible order
prices = prices[["date", "ticker", "open", "high", "low", "close", "volume"]]

prices.head(10)

,date,ticker,open,high,low,close,volume
0,2025-01-02,AAPL,247.577549,247.746638,240.506192,242.525162,55740700
1,2025-01-02,AMZN,222.029999,225.149994,218.190002,220.220001,33956600
2,2025-01-02,GOOGL,189.772645,191.116439,186.637147,188.558258,20370800
3,2025-01-02,META,587.363301,602.492600,585.470930,596.845276,12682300
4,2025-01-02,MSFT,421.452012,421.986845,410.874369,414.568604,16896500
5,2025-01-02,NVDA,135.955468,138.834530,134.585922,138.264709,198247200
6,2025-01-02,TSLA,390.100006,392.730011,373.040009,379.279999,109710700
7,2025-01-02,VOO,533.749081,535.245846,525.644626,529.258667,7142700
8,2025-01-03,AAPL,242.037827,242.853364,240.575812,242.037827,40244100
9,2025-01-03,AMZN,222.509995,225.360001,221.619995,224.190002,27515600


In [7]:
# Connect to (or create) a database file in this folder
conn = sqlite3.connect("equity_lab.db")

# Write the DataFrame to a table called "prices"
# if_exists="replace" overwrites if you re-run this cell
prices.to_sql("prices", conn, if_exists="replace", index=False)

conn.close()

print(f"Wrote {len(prices)} rows to equity_lab.db")

Wrote 2000 rows to equity_lab.db


In [10]:
conn = sqlite3.connect("equity_lab.db")

# Highest-volume day for each stock
query = """
SELECT ticker, MAX(volume) as peak_volume, date
FROM prices
GROUP BY ticker
ORDER BY peak_volume DESC
"""

peaks = pd.read_sql(query, conn)
conn.close()

peaks

,Ticker,peak_volume,Date
0,NVDA,818830900,2025-01-27 00:00:00
1,TSLA,287499800,2025-06-05 00:00:00
2,AAPL,184395900,2025-04-09 00:00:00
3,AMZN,166340800,2025-10-31 00:00:00
4,GOOGL,127490100,2025-05-07 00:00:00
5,META,88440100,2025-10-30 00:00:00
6,MSFT,70836100,2025-12-19 00:00:00
7,VOO,37062400,2025-12-18 00:00:00
